# EICC GRPO Training Notebook (Theme #3.1 World Modeling)

HF Space-ready workflow for two parallel lanes:

- **Lane A (Simple env)**: run training + compare in simulated backend.
- **Lane B (Sandbox env)**: run training + compare in sandbox backend (no Docker path).

Recommended run order (both lanes):
1. Clone repo
2. Install dependencies (Unsloth + TRL + peft + bitsandbytes)
3. Dry-run sanity check
4. Baseline evaluation
5. GRPO training (quick profile for ~1 hour)
6. Post-training compare evaluation (`trained_checkpoint` default)
7. Inspect reports (`baseline_report.json`, `trained_report.json`, `policy_used`)
8. Display mentor curves (`reward_curve_easy/medium/hard.png`)
9. (Optional) Two-seed reproducibility
10. (Optional) Mean ± std summary

Sandbox lane note:
- Start sandbox services without Docker using `python -m sandbox.launch_no_docker`
- Then run Step 6 sandbox compare command from another terminal in the same Space.

All artifacts land in `artifacts/...` and can be downloaded from the Space file browser.

In [ ]:
# Step 1: Clone your repo
!git clone https://github.com/anujkamaljain/OpenEnv-Customer-Support.git
%cd OpenEnv-Customer-Support

In [ ]:
# Step 2: Install training dependencies (clean path)
# If Colab asks for runtime restart, restart and rerun from Step 1.
%pip install -q -U pip setuptools wheel jedi
%pip install -q -U "pydantic>=2.12,<3" "click<8.2"
%pip install -q -U "trl==0.15.2" "transformers==4.51.3" datasets peft bitsandbytes accelerate matplotlib llm-blender mergekit
%pip install -q -U unsloth

In [ ]:
# Step 3: Verify training stack + environment (quick sanity check)
import importlib.metadata as im
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

print("unsloth:", im.version("unsloth"))
print("trl:", im.version("trl"))
print("transformers:", im.version("transformers"))

!python train.py --iterations 1 --episodes 1 --k 2 --dry-run

In [ ]:
# Step 4: Run baseline evaluation (before training)
# Lane A (simple env):
!python evaluate.py --policy baseline --episodes-per-difficulty 5 --output-dir artifacts/eval_simple

# Lane B (sandbox env) baseline should be run after sandbox services are started.
# See Step 6 sandbox instructions.

In [ ]:
# Step 5a: QUICK training (~1 hour target on A10/T4)
# After this finishes, artifacts/train/trained_adapter/ will exist and Step 6 can run.
!python -u train.py --iterations 6 --episodes 8 --k 2 --eval-episodes 1 --seed 7 --max-completion-length 96 --output-dir artifacts/train

# Step 5b: STRONGER training (if time remains)
# !python -u train.py --iterations 10 --episodes 15 --k 2 --eval-episodes 1 --seed 7 --max-completion-length 96 --output-dir artifacts/train

In [ ]:
# Step 6A: Post-training compare (Lane A: simple env)
# Mentor-friendly stage curves: 7 episodes per difficulty => 7 stages per graph.
!python -u evaluate.py --policy compare --compare-trained-policy trained_checkpoint --checkpoint-dir artifacts/train/trained_adapter --checkpoint-base-model Qwen/Qwen2.5-3B-Instruct --episodes-per-difficulty 7 --plot --output-dir artifacts/eval_simple

# Step 6B (Lane B: sandbox env in HF Space)
# 1) In another terminal inside the same Space, keep this running:
#    python -m sandbox.launch_no_docker
# 2) In this notebook/terminal, run:
# !OPENENV_SANDBOX_CLUSTER_URL=http://127.0.0.1 OPENENV_SANDBOX_CHAOS_URL=http://127.0.0.1:6660 \
#   python -u evaluate.py --policy compare --compare-trained-policy trained_checkpoint \
#   --checkpoint-dir artifacts/train/trained_adapter --checkpoint-base-model Qwen/Qwen2.5-3B-Instruct \
#   --episodes-per-difficulty 7 --plot --sandbox --output-dir artifacts/eval_sandbox

In [ ]:
import json
from pathlib import Path
TRACK = "simple"  # or "sandbox"
report_dir = Path("artifacts/eval_simple" if TRACK == "simple" else "artifacts/eval_sandbox")

b = json.loads((report_dir / "baseline_report.json").read_text())
t = json.loads((report_dir / "trained_report.json").read_text())
print("Track:", TRACK)
print("policy_used:", t.get("policy_used"))
print("Baseline raw:", b["avg_raw_reward"], "norm:", b["avg_normalized_reward"])
print("Trained  raw:", t["avg_raw_reward"], "norm:", t["avg_normalized_reward"])

In [ ]:
# Step 8: Display mentor graphs (easy/medium/hard)
from pathlib import Path
from IPython.display import Image, display

TRACK = "simple"  # or "sandbox"
curve_dir = Path("artifacts/eval_simple" if TRACK == "simple" else "artifacts/eval_sandbox")
curves = [
    curve_dir / "reward_curve_easy.png",
    curve_dir / "reward_curve_medium.png",
    curve_dir / "reward_curve_hard.png",
]
missing = [str(p) for p in curves if not p.exists()]
if missing:
    print("Run Step 6 with --plot first. Missing:")
    for p in missing:
        print(" -", p)
else:
    for curve in curves:
        print(curve.name)
        display(Image(filename=str(curve)))

In [ ]:
# Step 9 (optional): Two-seed reproducibility run
# Use this for final submission metrics (mean +/- std)
!python train.py --iterations 10 --episodes 15 --k 2 --seed 7 --max-completion-length 128 --output-dir artifacts/train_seed7
!python train.py --iterations 10 --episodes 15 --k 2 --seed 17 --max-completion-length 128 --output-dir artifacts/train_seed17

In [ ]:
# Step 10 (optional): Summarize two-seed metrics
from pathlib import Path
import json, math

paths = [
    Path("artifacts/train_seed7/trained_report.json"),
    Path("artifacts/train_seed17/trained_report.json"),
]
vals_norm, vals_raw = [], []
for p in paths:
    if p.exists():
        obj = json.loads(p.read_text(encoding="utf-8"))
        vals_norm.append(float(obj["avg_normalized_reward"]))
        vals_raw.append(float(obj.get("avg_raw_reward", 0.0)))

if len(vals_norm) == 2:
    mean_norm = sum(vals_norm) / 2
    mean_raw = sum(vals_raw) / 2
    std_norm = math.sqrt(sum((x - mean_norm) ** 2 for x in vals_norm) / 2)
    std_raw = math.sqrt(sum((x - mean_raw) ** 2 for x in vals_raw) / 2)
    print(f"avg_normalized_reward: {mean_norm:.4f} +/- {std_norm:.4f}")
    print(f"avg_raw_reward: {mean_raw:.4f} +/- {std_raw:.4f}")
else:
    print("Run Step 9 first to produce both seed reports.")